In [1]:
## -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from pandas import DataFrame, Series
from math import *
import collections
import time
import re
import itertools
import random
import pickle
import RNA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import multivariate_normal
from scipy.sparse import coo_matrix
import warnings
from scipy.spatial.distance import jensenshannon
from numba import njit, prange

import os
import json
mylen = np.vectorize(len)

warnings.filterwarnings("ignore")
%matplotlib inline

In [2]:
### Define functions

'''
Data sampling function

- sampling_dataframe: samples sequences from each round of a dataset according to their counts.
    - N: number of sequences to sample from each round.
    - random_state: seed for reproducibility.

- dataloader: loads the dataset based on the specified name, applies filtering based on length and rounds, and returns the processed dataset.
    - name: name of the dataset to load (e.g., 'N14', 'N20', etc.).
    - length_range: tuple specifying the minimum and maximum length of sequences to include (default is 'default', which applies predefined length filters based on the dataset).
    - N: number of sequences to sample from each round (default is 100000).
    - remove_rounds: list of rounds to remove from the dataset (default is 'default', which applies predefined round removals based on the dataset).
    - method: method for sampling sequences ('all' for random sampling, 'top' for top N sequences by count, 'none' for no sampling).
'''

def sampling_dataframe(ds, N, random_state=None):
    np.random.seed(random_state)
    ds_sampled = []
    for i in range(len(ds)):
        d = ds[i]
        seqs = d['seqs'].values
        counts = d['counts'].values
        if counts.sum() < N:
            print('Not enough counts in round {}: {}'.format(i+1, counts.sum()))
            ds_sampled.append(d)
            continue
        inds = np.random.choice(len(counts), size=N, replace=True, p=counts/counts.sum())
        seqs_sampled = [seqs[i] for i in inds]
        c = collections.Counter(seqs_sampled)
        seqs, counts = zip(*c.most_common())
        ds_sampled.append(DataFrame({'seqs': seqs, 'counts': counts, 'lens': mylen(seqs)}))
    return ds_sampled

def dataloader(name, length_range='default', N=100000, remove_rounds='default', random_state=None, method='all'):
    if name not in ['N14', 'N14_new', 'N14p', 'N14p_new', 'N20', 'N20p', 'single_N14', 'single_N14p', '2mer', '2mer-no-filter']:
        raise ValueError(f'Invalid name: {name}. Must be one of ["N14", "N14_new", "N14p", "N14p_new", "N20", "N20p", "single_N14", "single_N14p", "2mer", "2mer-no-filter"].')

    if name in ['N20', 'N20p']:
        Rs = [2, 4, 6, 8, 11, 13, 15, 17, 19, 22, 24, 26, 28, 29]

    elif name == 'N14p':
        Rs = [1, 2, 3, 5, 6, 8, 11, 13, 15, 17, 19, 22, 24, 26, 28, 29]

    elif name in ['N14', 'N14_new', 'N14p_new']:
        Rs = [1, 2, 3, 5, 6, 8, 10, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28, 29, 30]

    elif name == 'single_N14' or name == 'single_N14p':
        Rs = [1, 2, 3, 4, 5, 6]

    elif name == '2mer' or name == '2mer-no-filter':
        Rs = [1, 2, 3, 4, 5, 6]

    if name == 'single_N14':
        with open(f'data/filtered/single_filtered_blast.pkl', 'rb') as f:
            ds = pickle.load(f)
        ds = ds[:6]
    elif name == 'single_N14p':
        with open(f'data/filtered/single_filtered_blast.pkl', 'rb') as f:
            ds = pickle.load(f)
        ds = ds[6:]
    elif name == '2mer':
        with open(f'data/filtered/2mer_filtered_blast_no_repeats_norm.pkl', 'rb') as f:
            ds = pickle.load(f)
    elif name == '2mer-no-filter':
        with open(f'data/filtered/2mer_filtered_blast.pkl', 'rb') as f:
            ds = pickle.load(f)
    else:
        with open(f'data/filtered/{name}_filtered_blast.pkl', 'rb') as f:
            ds = pickle.load(f)

    if remove_rounds != 'default':
        if not (isinstance(remove_rounds, (list, tuple)) and all(r in Rs for r in remove_rounds)):
            raise ValueError(f'remove_rounds must be a list or tuple of integers in {Rs}.')
        remove_rounds = np.sort(list(set(remove_rounds)))
        remove_indices = [Rs.index(r) for r in remove_rounds]
        ds = [d for i, d in enumerate(ds) if i not in remove_indices]
        Rs = [r for r in Rs if r not in remove_rounds]
    else:
        pass

    if length_range != 'default':
        if not (isinstance(length_range, (list, tuple)) and len(length_range) == 2):
            raise ValueError('length_range must be a list or tuple of two integers.')
        min_len, max_len = length_range
        if not (isinstance(min_len, int) and isinstance(max_len, int) and min_len <= max_len):
            raise ValueError('length_range must contain two integers where the first is less than or equal to the second.')
        ds = [d[(d['lens'] >= min_len) & (d['lens'] <= max_len)].reset_index(drop=True) for d in ds]
    else:
        if name in ['N14', 'N14_new', 'N14p', 'N14p_new', 'single_N14', 'single_N14p']:
            ds = [d[(d['lens'] >= 18) & (d['lens'] <= 30)].reset_index(drop=True) for d in ds]
        elif name in ['N20', 'N20p']:
            ds = [d[(d['lens'] >= 25) & (d['lens'] <= 45)].reset_index(drop=True) for d in ds]
        elif name == '2mer' or name == '2mer-no-filter':
            ds = [d[(d['lens'] > 22) & (d['lens'] <= 30)].reset_index(drop=True) for d in ds]
    
    if method == 'all':
        ds = sampling_dataframe(ds, N, random_state=random_state)
    elif method == 'top':
        ds = [d.nlargest(N, 'counts').reset_index(drop=True) if len(d) >= N else d for d in ds]
    elif method == 'none':
        pass
    else:
        raise ValueError('method must be "all", "top", or "none".')

    return ds, Rs


'''
Functions for DataFrame processing

- concatenate_dataframes: concatenates a list of DataFrames into a single DataFrame and adds a 'label' column indicating the original round of each sequence.
    - ds: list of DataFrames to concatenate.

- deploy_df_by_counts: expands a DataFrame by replicating each row according to the value in the 'counts' column, effectively "deploying" the DataFrame based on counts.
    - df: DataFrame to deploy.
    - col_name: name of the column containing the counts (default is 'counts').
'''

def concatenate_dataframes(ds):
    lens = [len(d) for d in ds]
    inds_rounds = [np.repeat(i, lens[i]) for i in range(len(ds))]
    d = pd.concat(ds, ignore_index=True)
    d['label'] = np.concatenate(inds_rounds)
    return d

def deploy_df_by_counts(df, col_name='counts'):
    if col_name not in df.columns:
        raise KeyError(f'Column {col_name} not found in DataFrame.')
    counts = df[col_name].values
    if counts.sum() == 0:
        raise ValueError('Sum of counts is zero, cannot deploy DataFrame.')
    df = df.drop(columns=[col_name])
    values = np.zeros((counts.sum(), df.shape[1]), dtype=object)
    start = 0
    for i in range(len(counts)):
        end = start + counts[i]
        values[start:end] = df.iloc[i].values
        start = end
    return DataFrame(values, columns=df.columns)


'''
Functions for k-mer analysis

- make_kmers: generates all possible k-mers from a given set of bases.
    - bases: list of bases to use for generating k-mers
    - k: length of the k-mers to generate.

- normalize: normalizes the k-mer frequency data using the specified method.
    - X: DataFrame containing the k-mer frequencies to normalize.
    - method: method for normalization ('standard' for standard scaling, 'center' for centaring, None for no normalization).

- cal_kmer_freqs: calculates k-mer frequencies for a list of sequences.
    - sequences: list of sequences to analyze.
    - k: length of the k-mers to calculate frequencies for.
    - bases: list of bases to consider for k-mer generation (default is ['A', 'T', 'G', 'C']).
    - convert_freqs: whether to convert counts to frequencies by dividing by the number of k-mers in each sequence (default is True).
    - nonzero_only: whether to calculate frequencies only for k-mers that appear at least once in the dataset (default is False).

- make_kmer_dataframe: creates a DataFrame containing k-mer frequencies along with labels and counts for a given dataset.
    - ds: list of DataFrames for each round of the dataset.
    - k: length of the k-mers to calculate frequencies for.

- make_pca_dataframe: creates a DataFrame containing PCA-transformed k-mer frequencies along with labels and counts for a given dataset.
    - df: DataFrame containing k-mer frequencies and labels.
    - k: length of the k-mers to calculate frequencies for.
    - normalize_method: method for normalizing k-mer frequencies before PCA ('standard', 'center', or None).
    - return_pca: whether to return the PCA object along with the DataFrame (default is False).
    - use_precomputed_pca: whether to use a precomputed PCA object (default is False).
    - pca: precomputed PCA object to use if use_precomputed_pca is True (default is None).
    
- js_divergence_gaussians: calculates the Jensen-Shannon divergence between two multivariate Gaussian distributions, optionally weighted by counts.
    - X0: array of samples from the first distribution.
    - X1: array of samples from the second distribution.
    - w0: weights for the samples in X0 (default is None, which means equal weights).
    - w1: weights for the samples in X1 (default is None, which means equal weights).
    - base: logarithm base to use for the divergence calculation (default is 2).
    - n_samples: number of samples to use for estimating the divergence (default is 100000).
    - random_state: seed for reproducibility (default is None).

- make_distance_matrix: creates a distance matrix containing the Jensen-Shannon divergence between each pair of rounds in the dataset.
    - df: DataFrame containing PCA-transformed k-mer frequencies and labels for the dataset.
    - Rs: list of round labels corresponding to the rows in the DataFrame.
'''

def make_kmers(bases, k):
    return np.array([''.join(bp) for bp in itertools.product(bases, repeat = k)])

def normalize(X, method='standard'):
    if method == 'standard':
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    elif method == 'center':
        scaler = StandardScaler(with_std=False)
        X_scaled = scaler.fit_transform(X)
        X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    elif method == None:
        X_scaled = X.copy()
    else:
        raise ValueError("Unknown normalization method.")

    return X_scaled

def cal_kmer_freqs(sequences, k, bases=['A', 'T', 'G', 'C'], convert_freqs=True, nonzero_only=False):

    lens = mylen(sequences) - k + 1
    N = len(sequences)
    base2int = np.zeros(256, dtype=np.uint8)
    for i, b in enumerate(bases):
        base2int[ord(b)] = i

    if nonzero_only:
        unique_kmers = set()
        for seq in sequences:
            if len(seq) < k:
                continue
            for i in range(len(seq) - k + 1):
                kmer = seq[i:i+k]
                unique_kmers.add(kmer)
        kmers = np.array(sorted(unique_kmers))
        kmer_to_idx = {kmer: idx for idx, kmer in enumerate(kmers)}
        result = np.zeros((N, len(kmers)), dtype=np.float32)

        for i, seq in enumerate(sequences):
            if len(seq) < k:
                continue
            for j in range(len(seq) - k + 1):
                kmer = seq[j:j+k]
                idx = kmer_to_idx[kmer]
                result[i, idx] += 1
    else:
        kmers = np.array([''.join(bp) for bp in itertools.product(bases, repeat = k)])
        result = np.zeros((N, len(kmers)), dtype=np.float32)
        factor = np.int32(len(bases) ** (k - 1))

        for i, seq in enumerate(sequences):
            if len(seq) < k:
                continue
            arr = base2int[np.frombuffer(seq.encode('ascii'), dtype=np.uint8)].astype(np.int32)

            idx = np.int32(0)
            for j in range(k):
                idx = idx * len(bases) + arr[j]
            result[i, idx] += 1

            for j in range(k, len(arr)):
                idx = (idx % factor) * len(bases) + arr[j]
                result[i, idx] += 1

    if convert_freqs:
        result /= lens[:, None]
    return DataFrame(data=result, columns=kmers)

def make_kmer_dataframe(ds, k):
    df = concatenate_dataframes(ds)
    labels = df['label']
    counts = df['counts']
    df = cal_kmer_freqs(df['seqs'], k)
    df = pd.concat([df, labels, counts], axis=1)
    return df

def make_pca_dataframe(df, k, normalize_method='standard', return_pca=False, use_precomputed_pca=False, pca=None):
    df_norm = normalize(df.iloc[:,:-2], method=normalize_method)

    if use_precomputed_pca and pca is not None:
        pca = pca
    elif use_precomputed_pca and pca is None:
        raise ValueError('pca must be provided when use_precomputed_pca is True.')
    else:
        pca = PCA()
        pca.fit(df_norm.values)
    contribution_ratios = pd.DataFrame(pca.explained_variance_ratio_)
    contribution_ratios = contribution_ratios.cumsum().values
    contribution_ratios = contribution_ratios.flatten()
    df_pca = DataFrame(data=pca.transform(df_norm.values), columns=[f"PC{i+1}" for i in range(len(pca.components_))])

    num_below_threshold = 3 * 4**(k-1)

    df_new = pd.concat([df_pca.loc[:,'PC1':f'PC{num_below_threshold}'], df['label'], df['counts']], axis=1)
    if return_pca:
        return df_new, pca
    else:
        return df_new

def js_divergence_gaussians(X0, X1, w0=None, w1=None, base=2, n_samples=100000, random_state=None):

    if w0 is not None:
        w0 = w0 / np.sum(w0)
        mu0 = np.average(X0, axis=0, weights=w0)
        X0_centered = X0 - mu0
        cov0 = np.cov(X0_centered.T, aweights=w0, bias=False)
    else:
        mu0 = np.mean(X0, axis=0)
        cov0 = np.cov(X0.T, bias=False)

    if w1 is not None:
        w1 = w1 / np.sum(w1)
        mu1 = np.average(X1, axis=0, weights=w1)
        X1_centered = X1 - mu1
        cov1 = np.cov(X1_centered.T, aweights=w1, bias=False)
    else:
        mu1 = np.mean(X1, axis=0)
        cov1 = np.cov(X1.T, bias=False)

    rng = np.random.default_rng(random_state)

    p = multivariate_normal(mean=mu0, cov=cov0)
    q = multivariate_normal(mean=mu1, cov=cov1)

    Xp = p.rvs(size=n_samples, random_state=rng)
    log_p_Xp = p.logpdf(Xp)
    log_q_Xp = q.logpdf(Xp)
    log_m_Xp = np.logaddexp(log_p_Xp, log_q_Xp) - np.log(2.0)

    KL_pm = np.mean(log_p_Xp - log_m_Xp)

    Xq = q.rvs(size=n_samples, random_state=rng)
    log_p_Xq = p.logpdf(Xq)
    log_q_Xq = q.logpdf(Xq)
    log_m_Xq = np.logaddexp(log_p_Xq, log_q_Xq) - np.log(2.0)

    KL_qm = np.mean(log_q_Xq - log_m_Xq)

    return 0.5 * (KL_pm + KL_qm) / np.log(base)

def make_distance_matrix(df, Rs):
    distance_matrix = np.full((len(Rs), len(Rs)), np.nan)
    for i in range(len(Rs)):
        for j in range(i+1):
            X1 = df[df['label']==i].iloc[:,:-2].values
            w1 = df[df['label']==i]['counts'].values
            X2 = df[df['label']==j].iloc[:,:-2].values
            w2 = df[df['label']==j]['counts'].values
            distance_matrix[i][j] = js_divergence_gaussians(X1, X2, w1, w2)
            distance_matrix[j][i] = distance_matrix[i][j]
    return distance_matrix


'''
Functions for statistical analysis

- permutation_test: performs a permutation test to compare the average values of two groups of elements in a matrix
    - matrix: input square matrix to analyze.
    - N: number of permutations to perform (default is 100000).
    - order: number of diagonals to consider as the "close" group (default is 1).
    - side: which tail to consider for the p-value calculation ('right' for greater than observed difference, 'left' for less than observed difference, default is 'right').
    - pos: which part of the matrix to consider ('upper' for upper triangle, 'lower' for lower triangle, default is 'lower').
    - allow_diag: whether to include the diagonal elements in the "close" group (default is False).
'''

@njit(parallel=True)
def permutation_test(matrix, N=100000, order=1, side='right', pos='lower', allow_diag=False):
    if matrix.shape[0] != matrix.shape[1]:
        raise ValueError("Input matrix must be square")
    
    if order >= matrix.shape[0]:
        raise ValueError("order must be less than the size of the matrix")

    if pos == 'upper':
        matrix = matrix.T
    elif pos == 'lower':
        pass
    else:
        raise ValueError("pos must be 'upper' or 'lower'")
    
    if side not in ['right', 'left']:
        raise ValueError("side must be 'right' or 'left'")
    
    matrix = np.tril(matrix, k=-1) + np.tril(matrix).T
    R = matrix.shape[0]
    CR_counts = order * (2 * R - order - 1) / 2
    NCR_counts = (R - order) * (R - order - 1) / 2
    k = -1
    
    if allow_diag:
        CR_counts += R
        k = 0
    
    CR_counts = np.float64(CR_counts)
    NCR_counts = np.float64(NCR_counts)

    NCR_ave = np.tril(matrix, k=-(order+1)).sum() / NCR_counts
    CR_ave = (np.tril(matrix, k=k) - np.tril(matrix, k=-(order+1))).sum() / CR_counts
    obs_diff = NCR_ave - CR_ave

    count_r, count_l = 0, 0
    for n in prange(N):
        perm1 = np.random.permutation(R)
        perm2 = np.random.permutation(R)
        if  not allow_diag:
            perm1 = perm2
        perm_matrix = matrix[perm1][:, perm2]

        NCR_ave_perm = np.tril(perm_matrix, k=-(order+1)).sum() / NCR_counts
        CR_ave_perm = (np.tril(perm_matrix, k=k) - np.tril(perm_matrix, k=-(order+1))).sum() / CR_counts
        perm_diff = NCR_ave_perm - CR_ave_perm

        count_r += (perm_diff > obs_diff)
        count_l += (perm_diff < obs_diff)
    p_val_r = (count_r + 1) / (N + 1)
    p_val_l = (count_l + 1) / (N + 1)

    if side == 'right':
        return p_val_r
    elif side == 'left':
        return p_val_l


'''
Functions for RNA folding and structure analysis

- rna_folding: calculates the minimum free energy (MFE) structure and its corresponding MFE for each sequence in a DataFrame using the ViennaRNA package.
    - df: input DataFrame containing a 'seqs' column with RNA sequences.
    - temp: temperature in Celsius to use for RNA folding (default is 22).

- make_distance_matrix_mfe: creates a distance matrix based on the Jensen-Shannon divergence of the MFE distributions for each round in the dataset.
    - df: DataFrame containing MFE values and labels for each sequence.
    - Rs: list of round labels corresponding to the rows in the DataFrame.
    - mfe_range: tuple specifying the range of MFE values to consider for the histogram bins (default is (-30, 0)).
    - step: step size for the histogram bins (default is 0.5).

- structure_labeling: labels each position in an RNA secondary structure string according to its structural context
    - structure: string representing the RNA secondary structure in dot-bracket notation
'''

def rna_folding(df, temp=22):
    sequences = df['seqs'].values
    md = RNA.md()
    md.temperature = temp

    structures = []
    mfes = []
    for seq in sequences:
        structure, mfe = RNA.fold_compound(seq, md).mfe()
        structures.append(structure)
        mfes.append(mfe)
    df['sts'] = structures
    df['mfes'] = mfes
    return df

def make_distance_matrix_mfe(df, Rs, mfe_range=(-30, 0), step=0.5):
    if not (isinstance(mfe_range, (list, tuple)) and len(mfe_range) == 2):
        raise ValueError('mfe_range must be a list or tuple of two numbers.')
    distance_matrix = np.full((len(Rs), len(Rs)), np.nan)
    for i in range(len(Rs)):
        p, _ = np.histogram(df[df['label']==i]['mfes'].values, weights=df[df['label']==i]['counts'],
                            bins=np.arange(mfe_range[0], mfe_range[1]+step, step))
        p = p / p.sum()
        for j in range(i+1):
            q, _ = np.histogram(df[df['label']==j]['mfes'].values, weights=df[df['label']==j]['counts'],
                                bins=np.arange(mfe_range[0], mfe_range[1]+step, step))
            q = q / q.sum()
            distance_matrix[i][j] = jensenshannon(p, q, base=2)
            distance_matrix[j][i] = distance_matrix[i][j]
    return distance_matrix

def structure_labeling(structure):
    structure_arr = ['E']*len(structure)
    # Left stem
    left_stem_pattern = r'\('
    for span in [m.span() for m in re.finditer(left_stem_pattern, structure)]:
        structure_arr[span[0]:span[1]] = ['L']*(span[1]-span[0])
    
    # Right stem
    right_stem_pattern = r'\)'
    for span in [m.span() for m in re.finditer(right_stem_pattern, structure)]:
        structure_arr[span[0]:span[1]] = ['R']*(span[1]-span[0])
    
    # Hairpin loop
    hairpin_pattern = r'\({1}\.+\){1}'
    for span in [m.span() for m in re.finditer(hairpin_pattern, structure)]:
        structure_arr[span[0]+1:span[1]-1] = ['H']*(span[1]-span[0]-2)
    
    # Internal loop / Bulge
    internal_pattern = r'\({1}\.+\({1}|\){1}\.+\){1}'
    for span in [m.span() for m in re.finditer(internal_pattern, structure)]:
        structure_arr[span[0]+1:span[1]-1] = ['I']*(span[1]-span[0]-2)
    
    # External loop
    external_pattern = r'\){1}\.+\({1}'
    for span in [m.span() for m in re.finditer(external_pattern, structure)]:
        structure_arr[span[0]+1:span[1]-1] = ['X']*(span[1]-span[0]-2)
    
    return ''.join(structure_arr)


'''
Functions for complementary sequence analysis
Usage of sub-functions are omitted for brevity, see comments and code for details.

- count_subseq_occurrences_sparse: main function to count the occurrences of k-mers in a list of sequences and return a sparse matrix.
    - seqs_sub: list of sub_sequences (k-mers) to count occurrences for.
    - seqs: list of original sequences to analyze for k-mer occurrences.
    - bases: list of bases to consider for encoding and counting k-mers (default is ['A', 'T', 'G', 'C']).
    - return_sparse: whether to return the counts as a sparse matrix (default is True).
    - expand_unique: whether to expand the unique k-mer counts to a full sparse representation (default is True).
    
- count_kmer_overlaps: main function to create a sparse matrix representing shared k-mer overlaps between two sets of sequences.
    - seqs1: list of first set of sequences.
    - seqs2: list of second set of sequences.
    - k: length of k-mers to consider.
    - bases: list of bases to consider for encoding and counting k-mers (default is ['A', 'T', 'G', 'C']).
    - return_sparse: whether to return the overlaps as a sparse matrix (default is True).
'''

### encodes a list of sequences into a 2D array of integers based on a provided mapping
def _encode_sequences(seqs, base2int):
    n = len(seqs)
    if n == 0:
        return np.empty((0, 0), dtype=np.int8), np.empty(0, dtype=np.int32)
    lens = np.fromiter((len(s) for s in seqs), dtype=np.int32, count=n)
    max_len = int(lens.max())
    arr = np.zeros((n, max_len), dtype=np.int8)
    for i, s in enumerate(seqs):
        if len(s) == 0:
            continue
        arr[i, :len(s)] = base2int[np.frombuffer(s.encode('ascii'), dtype=np.uint8)]
    return arr, lens

### encodes a list of k-mer sequences into their corresponding integer codes based on a provided mapping
def _encode_kmer_codes(seqs_sub, base2int, base):
    codes = np.zeros(len(seqs_sub), dtype=np.int32)
    for i, s in enumerate(seqs_sub):
        arr = base2int[np.frombuffer(s.encode('ascii'), dtype=np.uint8)].astype(np.int32)
        idx = np.int32(0)
        for v in arr:
            idx = idx * base + v
        codes[i] = idx
    return codes

### counts the occurrences of k-mers in encoded sequences using a target mapping and returns the data, row indices, and column indices for constructing a sparse matrix
@njit(parallel=True)
def _count_subseq_occurrences_numba(seqs_encoded, seqs_lens, k, base, factor, target_map, num_unique):
    n = seqs_encoded.shape[0]
    nnz_per_seq = np.zeros(n, dtype=np.int64)
    for i in prange(n):
        L = seqs_lens[i]
        if L < k:
            continue
        counts = np.zeros(num_unique, dtype=np.int32)
        idx = np.int32(0)
        for j in range(k):
            idx = idx * base + seqs_encoded[i, j]
        u = target_map[idx]
        if u >= 0:
            counts[u] += 1
        for j in range(k, L):
            idx = (idx % factor) * base + seqs_encoded[i, j]
            u = target_map[idx]
            if u >= 0:
                counts[u] += 1
        c = 0
        for u in range(num_unique):
            if counts[u] != 0:
                c += 1
        nnz_per_seq[i] = c
    total = np.sum(nnz_per_seq)
    rows = np.empty(total, dtype=np.int64)
    cols = np.empty(total, dtype=np.int64)
    data = np.empty(total, dtype=np.int32)
    offsets = np.zeros(n + 1, dtype=np.int64)
    for i in range(n):
        offsets[i + 1] = offsets[i] + nnz_per_seq[i]
    for i in prange(n):
        L = seqs_lens[i]
        if L < k:
            continue
        counts = np.zeros(num_unique, dtype=np.int32)
        idx = np.int32(0)
        for j in range(k):
            idx = idx * base + seqs_encoded[i, j]
        u = target_map[idx]
        if u >= 0:
            counts[u] += 1
        for j in range(k, L):
            idx = (idx % factor) * base + seqs_encoded[i, j]
            u = target_map[idx]
            if u >= 0:
                counts[u] += 1
        offset = offsets[i]
        t = 0
        for u in range(num_unique):
            c = counts[u]
            if c != 0:
                rows[offset + t] = u
                cols[offset + t] = i
                data[offset + t] = c
                t += 1
    return data, rows, cols

### expands a sparse representation of unique k-mer counts to a full sparse representation by replicating counts for sequences that share the same unique k-mers.
def _expand_unique_sparse(rows_u, cols_u, data_u, inverse_indices, num_unique):
    order = np.argsort(inverse_indices)
    counts = np.bincount(inverse_indices, minlength=num_unique)
    offsets = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(counts)))
    entry_order = np.argsort(rows_u)
    rows_u = rows_u[entry_order]
    cols_u = cols_u[entry_order]
    data_u = data_u[entry_order]

    rows_list = []
    cols_list = []
    data_list = []
    pos = 0
    for u in range(num_unique):
        orig_rows = order[offsets[u]:offsets[u + 1]]
        if orig_rows.size == 0:
            continue
        start = pos
        while pos < rows_u.shape[0] and rows_u[pos] == u:
            pos += 1
        if start == pos:
            continue
        cols_slice = cols_u[start:pos]
        data_slice = data_u[start:pos]
        rows_block = np.repeat(orig_rows, cols_slice.shape[0])
        cols_block = np.tile(cols_slice, orig_rows.size)
        data_block = np.tile(data_slice, orig_rows.size)
        rows_list.append(rows_block)
        cols_list.append(cols_block)
        data_list.append(data_block)

    if not rows_list:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int32)
    rows = np.concatenate(rows_list)
    cols = np.concatenate(cols_list)
    data = np.concatenate(data_list)
    return data, rows, cols

def count_subseq_occurrences_sparse(seqs_sub, seqs, bases=('A', 'T', 'G', 'C'), return_sparse=True, expand_to_original=True):
    if len(seqs_sub) == 0:
        raise ValueError('seqs_sub is empty.')
    k = len(seqs_sub[0])
    if any(len(s) != k for s in seqs_sub):
        raise ValueError('All seqs_sub must have the same length k.')

    unique_subs, inverse_indices = np.unique(np.array(seqs_sub), return_inverse=True)
    num_unique = len(unique_subs)

    base2int = np.zeros(256, dtype=np.uint8)
    for i, b in enumerate(bases):
        base2int[ord(b)] = i
    base = np.int32(len(bases))
    factor = np.int32(base ** (k - 1))

    unique_codes = _encode_kmer_codes(unique_subs, base2int, base)
    target_map = np.full(base ** k, -1, dtype=np.int32)
    for u, code in enumerate(unique_codes):
        target_map[code] = u

    seqs_encoded, seqs_lens = _encode_sequences(seqs, base2int)
    data_u, rows_u, cols_u = _count_subseq_occurrences_numba(
        seqs_encoded, seqs_lens, k, base, factor, target_map, num_unique
    )

    if not expand_to_original:
        if return_sparse:
            S_unique = coo_matrix((data_u, (rows_u, cols_u)), shape=(num_unique, len(seqs)))
            return S_unique, unique_subs, inverse_indices
        return data_u, rows_u, cols_u, (num_unique, len(seqs)), unique_subs, inverse_indices

    data, rows, cols = _expand_unique_sparse(rows_u, cols_u, data_u, inverse_indices, num_unique)

    if return_sparse:
        return coo_matrix((data, (rows, cols)), shape=(len(seqs_sub), len(seqs)))
    return data, rows, cols, (len(seqs_sub), len(seqs))

### computes the overlap matrix between two sparse matrices represented in COO format, where the rows correspond to k-mers and the columns correspond to sequences.
@njit(parallel=True)
def _compute_overlap_matrix_numba(rows1, cols1, data1, rows2, cols2, data2, n1, n2, num_kmers):
    
    order1 = np.argsort(rows1)
    rows1_sorted = rows1[order1]
    cols1_sorted = cols1[order1]
    data1_sorted = data1[order1]
    
    order2 = np.argsort(rows2)
    rows2_sorted = rows2[order2]
    cols2_sorted = cols2[order2]
    data2_sorted = data2[order2]
    
    counts1 = np.zeros(num_kmers, dtype=np.int64)
    counts2 = np.zeros(num_kmers, dtype=np.int64)
    
    for r in rows1_sorted:
        counts1[r] += 1
    for r in rows2_sorted:
        counts2[r] += 1
    
    offsets1 = np.zeros(num_kmers + 1, dtype=np.int64)
    offsets2 = np.zeros(num_kmers + 1, dtype=np.int64)
    for i in range(num_kmers):
        offsets1[i + 1] = offsets1[i] + counts1[i]
        offsets2[i + 1] = offsets2[i] + counts2[i]
    
    total = np.int64(0)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        total += n1_entries * n2_entries
    
    result_rows = np.empty(total, dtype=np.int64)
    result_cols = np.empty(total, dtype=np.int64)
    result_data = np.empty(total, dtype=np.int32)
    
    kmer_offsets = np.zeros(num_kmers + 1, dtype=np.int64)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        kmer_offsets[kmer_idx + 1] = kmer_offsets[kmer_idx] + n1_entries * n2_entries
    
    for kmer_idx in prange(num_kmers):
        start1 = offsets1[kmer_idx]
        end1 = offsets1[kmer_idx + 1]
        start2 = offsets2[kmer_idx]
        end2 = offsets2[kmer_idx + 1]
        
        base_offset = kmer_offsets[kmer_idx]
        idx = 0
        for i in range(start1, end1):
            for j in range(start2, end2):
                result_rows[base_offset + idx] = cols1_sorted[i]
                result_cols[base_offset + idx] = cols2_sorted[j]
                result_data[base_offset + idx] = data1_sorted[i] * data2_sorted[j]
                idx += 1
    
    return result_data, result_rows, result_cols

### Binary version of the overlap matrix computation, where the data values are set to 1 for any overlap regardless of the original counts.
@njit(parallel=True)
def _compute_overlap_matrix_numba_binary(rows1, cols1, data1, rows2, cols2, data2, n1, n2, num_kmers):

    order1 = np.argsort(rows1)
    rows1_sorted = rows1[order1]
    cols1_sorted = cols1[order1]
    
    order2 = np.argsort(rows2)
    rows2_sorted = rows2[order2]
    cols2_sorted = cols2[order2]
    
    counts1 = np.zeros(num_kmers, dtype=np.int64)
    counts2 = np.zeros(num_kmers, dtype=np.int64)
    
    for r in rows1_sorted:
        counts1[r] += 1
    for r in rows2_sorted:
        counts2[r] += 1
    
    offsets1 = np.zeros(num_kmers + 1, dtype=np.int64)
    offsets2 = np.zeros(num_kmers + 1, dtype=np.int64)
    for i in range(num_kmers):
        offsets1[i + 1] = offsets1[i] + counts1[i]
        offsets2[i + 1] = offsets2[i] + counts2[i]
    
    total = np.int64(0)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        total += n1_entries * n2_entries
    
    result_rows = np.empty(total, dtype=np.int64)
    result_cols = np.empty(total, dtype=np.int64)
    result_data = np.ones(total, dtype=np.int32)
    
    kmer_offsets = np.zeros(num_kmers + 1, dtype=np.int64)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        kmer_offsets[kmer_idx + 1] = kmer_offsets[kmer_idx] + n1_entries * n2_entries
    
    for kmer_idx in prange(num_kmers):
        start1 = offsets1[kmer_idx]
        end1 = offsets1[kmer_idx + 1]
        start2 = offsets2[kmer_idx]
        end2 = offsets2[kmer_idx + 1]
        
        base_offset = kmer_offsets[kmer_idx]
        idx = 0
        for i in range(start1, end1):
            for j in range(start2, end2):
                result_rows[base_offset + idx] = cols1_sorted[i]
                result_cols[base_offset + idx] = cols2_sorted[j]
                idx += 1
    
    return result_data, result_rows, result_cols

def count_kmer_overlaps(seqs1, seqs2, k, bases=('A', 'T', 'G', 'C'), return_sparse=True, binary=False):
    if len(seqs1) == 0 or len(seqs2) == 0:
        if return_sparse:
            return coo_matrix((len(seqs1), len(seqs2)))
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), (len(seqs1), len(seqs2))
    
    lens1 = np.fromiter((len(s) for s in seqs1), dtype=np.int32, count=len(seqs1))
    lens2 = np.fromiter((len(s) for s in seqs2), dtype=np.int32, count=len(seqs2))
    if np.any(lens1 < k) or np.any(lens2 < k):
        raise ValueError('All sequences must have length at least k.')
    
    kmers1 = set()
    for s in seqs1:
        for i in range(len(s) - k + 1):
            kmers1.add(s[i:i + k])
    
    kmers2 = set()
    for s in seqs2:
        for i in range(len(s) - k + 1):
            kmers2.add(s[i:i + k])
    
    common_kmers = list(kmers1.intersection(kmers2))
    
    if not common_kmers:
        if return_sparse:
            return coo_matrix((len(seqs1), len(seqs2)))
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), (len(seqs1), len(seqs2))
    
    data1, rows1, cols1, shape1, unique_kmers, _ = count_subseq_occurrences_sparse(
        common_kmers, seqs1, bases=bases, return_sparse=False, expand_to_original=False
    )
    data2, rows2, cols2, shape2, _, _ = count_subseq_occurrences_sparse(
        common_kmers, seqs2, bases=bases, return_sparse=False, expand_to_original=False
    )
    
    if len(data1) == 0 or len(data2) == 0:
        if return_sparse:
            return coo_matrix((len(seqs1), len(seqs2)))
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), (len(seqs1), len(seqs2))
    
    if binary:
        result_data, result_rows, result_cols = _compute_overlap_matrix_numba_binary(
            rows1, cols1, data1, rows2, cols2, data2, len(seqs1), len(seqs2), shape1[0]
        )
    else:
        result_data, result_rows, result_cols = _compute_overlap_matrix_numba(
            rows1, cols1, data1, rows2, cols2, data2, len(seqs1), len(seqs2), shape1[0]
        )

    if return_sparse:
        if not binary:
            S = coo_matrix((result_data, (result_rows, result_cols)), shape=(len(seqs1), len(seqs2)))
            S.sum_duplicates()
        if binary:
            hashes = result_rows * len(seqs2) + result_cols
            unique_hashes, counts = np.unique(hashes, return_counts=True)
            data = np.ones_like(counts, dtype=np.int32)
            S = coo_matrix((data, (unique_hashes // len(seqs2), unique_hashes % len(seqs2))), shape=(len(seqs1), len(seqs2)))
        return S
    
    return result_data, result_rows, result_cols, (len(seqs1), len(seqs2))

In [3]:
"""
Functions for additional analysis for revised manuscript

- mask_triangle: masks either the upper or lower triangle of a matrix with a specified value
    - matrix: input square matrix to mask.
    - upper: whether to mask the upper triangle (True) or lower triangle (False, default is True).
    - mask_number: value to use for masking the triangle (default is np.nan).

- base_pairs: identifies the base pairs in a dot-bracket notation
    - st: input string in dot-bracket notation.

- structure_labeling_cofold: labels each position in a cofolded RNA secondary structure string
    -st: input string in dot-bracket notation.

- generate_random_sequences: generates a list of random sequences based on specified base frequencies
    - N: number of sequences to generate.
    - length: length of each sequence.
    - base: list of bases to use for sequence generation (default is ['A', 'T', 'G', 'C']).
    - base_freqs: optional 1D or 2D array specifying the frequencies of each base at each position (default is None, which uses uniform frequencies).
    - random_state: optional seed for the random number generator (default is None).

- extract_overlap_regions: extracts the overlapping regions between two sets of sequences based on a sparse matrix representation of shared k-mers
    - seqs1: list of first set of sequences.
    - seqs2: list of second set of sequences.
    - S: sparse matrix representing shared k-mers between seqs1 and seqs2
    - k: length of the k-mers used to define overlaps
"""

def mask_triangle(matrix, upper=True, mask_number=np.nan):
    masked_matrix = np.full_like(matrix, mask_number)
    for i in range(matrix.shape[0]):
        for j in range(i, matrix.shape[1]):
            if upper:
                masked_matrix[i][j] = matrix[i][j]
            else:
                masked_matrix[j][i] = matrix[j][i]
    return masked_matrix

def base_pairs(st):
    pairs = []
    stack = []
    for i, char in enumerate(st):
        if char == '(':
            stack.append(i)
        elif char == ')':
            if stack:
                j = stack.pop()
                pairs.append((j, i))
    return pairs


def structure_labeling_cofold(st):
    if '&' not in st:
        labeled_st = np.full(len(st), 'E', dtype='U1')
        for match in re.finditer(r'(?<=[\(\[\{\)\]\}])\.+(?=[\(\[\{\)\]\}])', st):
            labeled_st[match.start():match.end()] = 'T'
        for i, char in enumerate(st):
            if char in '([{<)]}>':
                labeled_st[i] = 'S'
        return ''.join(labeled_st)
    else:
        connect_ind = st.find('&')
        intermolecular_pairs = []
        intramolecular_pairs = []

        stack = []
        for i, char in enumerate(st):
            if char in '([{<':
                stack.append(i)
            elif char in ')]}>':
                if stack:
                    j = stack.pop()
                    if j < connect_ind and i > connect_ind:
                        intermolecular_pairs.append((j, i))
                    else:
                        intramolecular_pairs.append((j, i))
        intermolecular_pairs.sort(key=lambda x: (min(x), max(x)))
        intermolecular_pairs = np.array(intermolecular_pairs)
        intramolecular_pairs.sort(key=lambda x: (min(x), max(x)))
        intramolecular_pairs = np.array(intramolecular_pairs)

        internal_structure_inds = []
        for match in re.finditer(r'(?<=[\(\[\{\)\]\}])\.+(?=[\(\[\{\)\]\}])', st):
            internal_structure_inds.append((match.start(), match.end()))

        labeled_st = np.full(len(st), 'E', dtype='U1')
        labeled_st[connect_ind] = '&'
        for pair in intermolecular_pairs:
            labeled_st[pair[0]] = 'I'
            labeled_st[pair[1]] = 'I'
        for pair in intramolecular_pairs:
            labeled_st[pair[0]] = 'S'
            labeled_st[pair[1]] = 'S'
        for ind in internal_structure_inds:
            labeled_st[ind[0]:ind[1]] = 'T'
        labeled_st = ''.join(labeled_st)
        return labeled_st

def generate_random_sequences(N, length, base=['A', 'T', 'G', 'C'], base_freqs=None, random_state=None):
    if random_state is not None:
        random.seed(random_state)
    if base_freqs is not None:
        base_freqs = np.array(base_freqs)
        shape = base_freqs.shape
        if len(shape) > 2:
            raise ValueError("base_freqs should be a 1D or 2D array.")
        elif shape[0] != len(base):
            raise ValueError("Length of axis 0 of base_freqs should match the length of base.")
        elif len(shape) == 2 and shape[1] != length:
            raise ValueError("Length of axis 1 of base_freqs should match the specified length.")
        if len(shape) == 1:
            base_freqs = np.tile(base_freqs[:, np.newaxis], (1, length))
        base_freqs = base_freqs / base_freqs.sum(axis=0)
    else:
        base_freqs = np.full((len(base), length), 1/len(base))
    
    sequences = []
    for _ in range(N):
        sequence = []
        for i in range(length):
            probs = base_freqs[:, i]
            b = random.choices(base, weights=probs, k=1)[0]
            sequence.append(b)
        sequences.append(''.join(sequence))
    return sequences

def extract_overlap_regions(seqs1, seqs2, S, k):

    if hasattr(S, 'tocoo'):
        S_coo = S.tocoo()
    else:
        S_coo = S
    
    overlap_regions = []
    
    for row_idx, col_idx in zip(S_coo.row, S_coo.col):
        seq1 = seqs1[row_idx]
        seq2 = seqs2[col_idx]
        
        kmers1_positions = {}
        for pos in range(len(seq1) - k + 1):
            kmer = seq1[pos:pos + k]
            if kmer not in kmers1_positions:
                kmers1_positions[kmer] = []
            kmers1_positions[kmer].append(pos)
        
        kmers2_positions = {}
        for pos in range(len(seq2) - k + 1):
            kmer = seq2[pos:pos + k]
            if kmer not in kmers2_positions:
                kmers2_positions[kmer] = []
            kmers2_positions[kmer].append(pos)
        
        overlap_list = []
        for kmer in kmers1_positions:
            if kmer in kmers2_positions:
                for pos1 in kmers1_positions[kmer]:
                    for pos2 in kmers2_positions[kmer]:
                        overlap_list.append({
                            'kmer': kmer,
                            'seq1_start': pos1,
                            'seq1_end': pos1 + k,
                            'seq2_start': pos2,
                            'seq2_end': pos2 + k
                        })
        
        overlap_regions.append(tuple(overlap_list))
    
    return overlap_regions

"""
- overlap_unpairing_filtering: filters a sparse matrix of overlaps based on accesibility of the overlapping regions
    - S: sparse matrix representing shared k-mers between two sets of sequences
    - seqs: list of sequences corresponding to the rows and columns of S
    - overlap_regions: list of overlapping regions extracted from the sequences
    - k: length of the k-mers used to define overlaps
    - sts: optional list of secondary structures for the sequences (if not provided, they will be computed)
    - temp: temperature for RNA folding (default is 22.0)
    - unpair_threshold: threshold for the number of unpaired bases in the overlapping region (default is k)
    - target: specifies whether to filter based on rows, columns, or both

- random_filtering: randomly selects a specified number of overlaps from the sparse matrix
    - S: sparse matrix representing shared k-mers between two sets of sequences
    - overlap_regions: list of overlapping regions extracted from the sequences
    - N: number of overlaps to randomly select
"""


def overlap_unpairing_filtering(S, seqs, overlap_regions, k, sts=None, temp=22.0, unpair_threshold=None, target='row'):
    if target not in ['row', 'col', 'both']:
        raise ValueError("Target must be either 'row', 'col', or 'both'.")

    if unpair_threshold is None:
        unpair_threshold = k
    
    if hasattr(S, 'tocoo'):
        S = S.tocoo()

    md = RNA.md()
    md.temperature = temp

    if sts is None:
        sts = np.zeros(len(seqs), dtype=object)
        for i in range(len(seqs)):
            seq = seqs[i]
            st, _ = RNA.fold_compound(seq, md).mfe()
            st = structure_labeling_cofold(st)
            sts[i] = st

    if target in ['row', 'both']:
        filter_r = np.full(S.nnz, False)
        for r, c, reg, j in zip(S.row, S.col, overlap_regions, np.arange(S.nnz)):
            len_r = len(seqs[r])
            for region in reg:
                ovlp = (len_r-region['seq1_end'], len_r-region['seq1_start'])
                unpair_r = sts[r][ovlp[0]:ovlp[1]].count('E')
                if unpair_r >= unpair_threshold:
                    filter_r[j] = True
                    break
                
    if target in ['col', 'both']:
        filter_c = np.full(S.nnz, False)
        for r, c, reg, j in zip(S.row, S.col, overlap_regions, np.arange(S.nnz)):
            len_c = len(seqs[c])
            for region in reg:
                ovlp = (region['seq2_start'], region['seq2_end'])
                unpair_c = sts[c][ovlp[0]:ovlp[1]].count('E')
                if unpair_c >= unpair_threshold:
                    filter_c[j] = True
                    break
    
    if target == 'row':
        filter = filter_r
    elif target == 'col':
        filter = filter_c
    elif target == 'both':
        filter = filter_r & filter_c

    rows, cols, data = S.row, S.col, S.data
    filtered_rows = rows[filter]
    filtered_cols = cols[filter]
    filtered_data = data[filter]
    filtered_overlap_regions = [overlap_regions[i] for i in np.where(filter)[0]]

    return coo_matrix((filtered_data, (filtered_rows, filtered_cols)), shape=S.shape), filtered_overlap_regions

def random_filtering(S, overlap_regions, N, random_state=None):
    if random_state is not None:
        random.seed(random_state)
    
    if hasattr(S, 'tocoo'):
        S = S.tocoo()
    
    if N >= S.nnz:
        return S, overlap_regions
    
    inds = list(range(S.nnz))
    random.shuffle(inds)
    selected_inds = inds[:N]
    selected_inds.sort()

    rows, cols, data = S.row, S.col, S.data
    filtered_rows = rows[selected_inds]
    filtered_cols = cols[selected_inds]
    filtered_data = data[selected_inds]
    filtered_overlap_regions = [overlap_regions[i] for i in selected_inds]

    return coo_matrix((filtered_data, (filtered_rows, filtered_cols)), shape=S.shape), filtered_overlap_regions

"""
- pre_cleaving: simulates the cleavage of sequences based on specified probabilities for cleavage
    - seqs: list of sequences
    - lens: list of sequence lengths
    - labels: list of sequence labels
    - p_cleave: probability of cleavage at each position
    - p_cycphos: probability of forming a cyclic phosphate at 3'-end

- terminal_free_space: calculates the number of unpaired bases at the ends of two sequences in RNA-RNA complexes
    - st1: secondary structure of the first sequence in dot-bracket notation
    - st2: secondary structure of the second sequence in dot-bracket notation
    - ovlp: dictionary containing the overlap information between the two sequences
"""

def pre_cleaving(seqs, lens, labels, p_cleave, p_cycphos):
    for i, seq in enumerate(seqs):
        len_seq = len(seq)
        label = labels[i]
        if len_seq < 2:
            continue
        randoms = np.random.rand(len_seq - 1)
        inds = np.where(randoms < p_cleave)[0]
        if len(inds) == 0:
            continue

        cleaved_seqs = []
        cleaved_lens = []
        cleaved_labels = []
        start = 0
        for ind in inds:
            cleaved_seqs.append(seq[start:ind + 1])
            cleaved_lens.append(len(seq[start:ind + 1]))
            cleaved_labels.append('cyclic-phosphate') if np.random.rand() < p_cycphos else cleaved_labels.append('phosphate')
            start = ind + 1
        cleaved_seqs.append(seq[start:])
        cleaved_lens.append(len(seq[start:]))
        cleaved_labels.append(label)

        del seqs[i], lens[i], labels[i]
        seqs.extend(cleaved_seqs)
        lens.extend(cleaved_lens)
        labels.extend(cleaved_labels)
    return seqs, lens, labels

def terminal_free_space(st1, st2, ovlp):
    len1, len2 = len(st1), len(st2)
    reg1 = (len1-ovlp['seq1_end'], len1-ovlp['seq1_start'])
    reg2 = (ovlp['seq2_start'], ovlp['seq2_end'])

    g1, g2 = None, None

    m1 = re.search(r'\)+', st1[::-1])
    if m1 and reg1[1] < len1 - m1.start():
        n1 = m1.start()
        pairs = np.array(base_pairs(st1))
        p1 = pairs[pairs[:, 1] == len1 - m1.start() - 1, 0][0]
        g1 = int(p1 - reg1[1])
    else:
        n1 = len1 - reg1[1]

    m2 = re.search(r'\(+', st2)
    if m2 and reg2[0] > m2.start():
        n2 = m2.start()
        pairs = np.array(base_pairs(st2))
        p2 = pairs[pairs[:, 0] == m2.start(), 1][0]
        g2 = int(reg2[0] - p2 - 1)
    else:
        n2 = reg2[0]

    return n1, n2, g1, g2

"""
- ovlp_filtering: filters overlapping regions based on the accessibility of the overlapping regions
    - st1: secondary structure of the first sequence in dot-bracket notation
    - st2: secondary structure of the second sequence in dot-bracket notation
    - ovlps: list of overlapping regions between the two sequences
    - thr: threshold for the proportion of unpaired bases in the overlapping region (default is 1.0)

- reaction: simulates the ligation reaction between 2 sequences
    - df: dataframe containing sequences, labels, secondary structures, and counts
    - S: sparse matrix representing shared k-mers between sequences
    - overlap_regions: list of overlapping regions extracted from the sequences
    - num_reaction: number of reactions to simulate (default is 100000)
    - p_cleave: probability of cleavage at each position (default is 0.02)
    - p_cycphos: probability of forming a cyclic phosphate at 3'-end (default is 0.8)
    - gamma: parameter for the calculation of reaction probabilities (default is 2.0)
"""


def ovlp_filtering(st1, st2, ovlps, thr=1.0):
    ovlps_new = ()
    len1, len2 = len(st1), len(st2)
    for ovlp in ovlps:
        reg1 = (len1-ovlp['seq1_end'], len1-ovlp['seq1_start'])
        reg2 = (ovlp['seq2_start'], ovlp['seq2_end'])
        space1 = st1[reg1[0]:reg1[1]]
        space2 = st2[reg2[0]:reg2[1]]
        if space1.count('E') >= len(space1)*thr and space2.count('E') >= len(space2)*thr:
            ovlps_new += (ovlp,)
    if len(ovlps_new) == 0:
        return ()
    
    elif len(ovlps_new) == 1:
        return ovlps_new
    
    else:
        ovlps = ovlps_new
        ovlps_new = ()
        ovlp = ovlps[0]
        start1, start2, end1, end2, kmer = ovlp['seq1_start'], ovlp['seq2_start'], ovlp['seq1_end'], ovlp['seq2_end'], ovlp['kmer']
        ex_l = 1
        for ovlp in ovlps[1:]:
            start1_n, start2_n, end1_n, end2_n, kmer_n = ovlp['seq1_start'], ovlp['seq2_start'], ovlp['seq1_end'], ovlp['seq2_end'], ovlp['kmer']
            if start1_n - start1 == ex_l and start2_n - start2 == ex_l:
                end1, end2 = end1_n, end2_n
                kmer += kmer_n[-1]
                ex_l += 1
            else:
                ovlps_new += ({'kmer': kmer, 'seq1_start': start1, 'seq1_end': end1, 'seq2_start': start2, 'seq2_end': end2},)
                start1, start2, end1, end2, kmer = start1_n, start2_n, end1_n, end2_n, kmer_n
                ex_l = 1
        ovlps_new += ({'kmer': kmer, 'seq1_start': start1, 'seq1_end': end1, 'seq2_start': start2, 'seq2_end': end2},)
        return ovlps_new

def reaction(df, S, overlap_regions, num_reaction=100000, p_cleave=0.02, p_cycphos=0.8, gamma=2.0):

    seqs = df['seqs'].values
    labels = df['labels'].values
    labeled_sts = df['labeled_sts'].values
    sts = df['sts'].values
    counts = df['counts'].values

    weights = [counts[r] * counts[c] / 2 if r != c else counts[r] * (counts[r] - 1) for r, c in zip(S.row, S.col)]
    weights = np.array(weights)
    weights = weights / weights.sum()
    counts_new = counts.copy()

    new_seqs = []
    params = []
    labels_new = []
    total_reaction = 0
    while total_reaction < num_reaction:
        ind = np.random.choice(S.nnz, p=weights)
        r, c = S.row[ind], S.col[ind]
        cr = counts_new[r] - 1
        cc = counts_new[c] - 1
        if cr < 0 or cc < 0:
            continue

        total_reaction += 1
        st1, st2 = labeled_sts[r], labeled_sts[c]
        ovlps = overlap_regions[ind]
        ovlps = ovlp_filtering(st1, st2, ovlps, thr=1.0)
        if len(ovlps) == 0:
            continue
        ovlp = ovlps[np.argmax([len(ovlp['kmer']) for ovlp in ovlps])]

        st1, st2 = sts[r], sts[c]
        label1, label2 = labels[r], labels[c]
        seq1, seq2 = seqs[r], seqs[c]
        n1, n2, g1, g2 = terminal_free_space(st1, st2, ovlp)
        if label1 == 'phosphate' and n1 < 1:
            continue

        cleaves = np.array([np.random.rand() < p_cleave * p_cycphos for _ in range(n1)])
        l = np.where(cleaves)[0][0] if np.any(cleaves) else n1

        if label1 == 'phosphate' and l == n1:
            continue
        if g1 is not None and g2 is not None:
            continue
        if (g1 is None and g2 is None) and l + n2 < 4:
            continue
        if g1 is not None or g2 is not None:
            g = g1 if g1 is not None else g2
            if g >= 2:
                continue

        prob = (1 + l) ** (-gamma) * (1 + n2) ** (-gamma)
        if np.random.rand() > prob:
            continue

        new_seq = seq1[:len(seq1) - n1 + l] + seq2
        new_seqs.append(new_seq)
        params.append((n1, n2, g1, g2, int(l)))
        labels_new.append(label2)
        counts_new[r] = cr
        counts_new[c] = cc

    df['counts'] = counts_new
    df_prod = DataFrame({'seqs': new_seqs, 'lens': mylen(new_seqs), 'counts': np.ones(len(new_seqs), dtype=int), 'labels': labels_new, 'params': params})

    return df, df_prod

#### Response 2-1 & Table S1: Carryover correction
- name: {'N14', 'N14p', 'N20', 'N20p', 'N14_new', 'N14p_new'}
- k: number of nucleotide
- c: carryover rate
- N: number of sampling

In [ ]:
### Response 2-1: Carryover correction for 3-mer distribution analysis.

name = 'N14p_new'
k = 3
c = 0.2
N = 10

distance_matrix_set = []
for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=100000)
    df = make_kmer_dataframe(ds, k)
    df = make_pca_dataframe(df, k)

    params = []
    for i in range(len(Rs)):
        X = df[df['label']==i].iloc[:,:-2].values
        w = df[df['label']==i]['counts'].values
        w = w / w.sum()
        mu = np.average(X, axis=0, weights=w)
        X_centered = X - mu
        cov = np.cov(X_centered.T, aweights=w, bias=False)
        params.append((mu, cov))

    params_new = [params[0]]
    for i in range(len(Rs)-1):
        alpha = c**(Rs[i+1] - Rs[i])
        mu_prev, cov_prev = params[i]
        mu, cov = params[i+1]

        mu_new = (mu - alpha * mu_prev) / (1 - alpha)
        cov_new = (cov - alpha * cov_prev) / (1 - alpha) - alpha * (mu - mu_prev).reshape(-1, 1) @ (mu - mu_prev).reshape(1, -1) / (1 - alpha)**2

        if np.diag(cov_new).min() <= 0:
            print(f'Warning: Non-positive variance encountered in covariance matrix for {name} at iteration {n+1}. Adjusting to ensure positive definiteness.')
        params_new.append((mu_new, cov_new))

    rng = np.random.default_rng(None)
    distance_matrix = np.full((len(Rs), len(Rs)), np.nan)
    for i in range(len(Rs)):
        p = multivariate_normal(mean=params_new[i][0], cov=params_new[i][1])
        for j in range(i+1):
            if i == j:
                distance_matrix[i, j] = 0.0
                continue
            q = multivariate_normal(mean=params_new[j][0], cov=params_new[j][1])

            Xp = p.rvs(size=100000, random_state=rng)
            log_p_Xp = p.logpdf(Xp)
            log_q_Xp = q.logpdf(Xp)
            log_m_Xp = np.logaddexp(log_p_Xp, log_q_Xp) - np.log(2.0)
            KL_pm = np.mean(log_p_Xp - log_m_Xp)

            Xq = q.rvs(size=100000, random_state=rng)
            log_p_Xq = p.logpdf(Xq)
            log_q_Xq = q.logpdf(Xq)
            log_m_Xq = np.logaddexp(log_p_Xq, log_q_Xq) - np.log(2.0)
            KL_qm = np.mean(log_q_Xq - log_m_Xq)

            distance_matrix[i, j] = 0.5 * (KL_pm + KL_qm) / np.log(2.0)
            distance_matrix[j, i] = distance_matrix[i, j]
    distance_matrix_set.append(distance_matrix)
    
    p_val = permutation_test(distance_matrix, N=100000, order=1, side='right')
    sign = '*' if p_val < 0.05 else ''
    print(f'Elapsed time for {name} iteration {n+1}/{N}: {time.time() - start:.2f} seconds, p-value: {p_val:.5f}{sign}')
distance_matrix_set = np.array(distance_matrix_set)

result = {
    'distance_matrix_set': distance_matrix_set,
    'Rs': Rs,
    'c': c,
    'k': k,
    'N': N
}

np.savez_compressed(f'data/processed/review/carryover/{name}_k{k}_distance_matrix_c{c}.npz', **result)

p_val = permutation_test(distance_matrix_set.mean(axis=0), N=100000, order=1, side='right')
sign = '*' if p_val < 0.05 else ''
print(f'\nAverage distance matrix, p-value: {p_val:.5f}{sign}')

In [ ]:
### Response 2-1: Carryover correction for MFE distribution analysis.

c = 0.2
N = 10

for name in ['N14', 'N14_new', 'N14p', 'N14p_new', 'N20', 'N20p']:
    if name in ['N14', 'single_N14', 'N14p', 'single_N14p', 'N14_new', 'N14p_new']:
        mfe_range = (-30, 0)
    else:
        mfe_range = (-40, 0)
    step = 0.5

    distance_matrix_set = []
    for n in range(N):
        start = time.time()
        ds, Rs = dataloader(name, N=100000)
        df = concatenate_dataframes(ds)
        df = rna_folding(df)
        df = df[df['mfes']!=0].reset_index(drop=True)

        ps = []
        for i in range(len(Rs)):
            p, _ = np.histogram(df[df['label']==i]['mfes'].values, weights=df[df['label']==i]['counts'],
                                bins=np.arange(mfe_range[0], mfe_range[1]+step, step))
            p = p / p.sum()
            ps.append(p)

        ps_new = [ps[0]]
        for i in range(len(Rs)-1):
            alpha = c**(Rs[i+1] - Rs[i])
            p_prev = ps[i]
            p = ps[i+1]

            p_new = (p - alpha * p_prev) / (1 - alpha)
            p_new[p_new < 0] = 0
            p_new = p_new / p_new.sum()
            ps_new.append(p_new)

        distance_matrix = np.full((len(Rs), len(Rs)), np.nan)
        for i in range(len(Rs)):
            p = ps_new[i]
            for j in range(i+1):
                if i == j:
                    distance_matrix[i, j] = 0.0
                    continue
                q = ps_new[j]

                distance_matrix[i, j] = jensenshannon(p, q, base=2)
        distance_matrix_set.append(distance_matrix)
        print(f'Elapsed time for {name} iteration {n+1}/{N}: {time.time() - start:.2f} seconds')
    distance_matrix_set = np.array(distance_matrix_set)

    result = {
        'distance_matrix_set': distance_matrix_set,
        'Rs': Rs,
        'mfe_range': mfe_range,
        'c': c,
        'N': N,
    }

    np.savez_compressed(f'data/processed/review/carryover/{name}_distance_matrix_mfe_c{c}.npz', **result)

In [4]:
names = ['N14', 'N14p', 'N20', 'N20p', 'N14_new', 'N14p_new']
key = 'kmer' # or 'mfe'
k = 3
c = 0.2

print('p-Value for Original divergence:')
for name in names:
    if key == 'kmer':
        distance_matrix_set = np.load(f'data/processed/distance_matrix/{name}_k{k}_distance_matrix.npy')
    elif key == 'mfe':
        distance_matrix_set = np.load(f'data/processed/structure/{name}_distance_matrix.npy')
    dm = np.mean(distance_matrix_set, axis=0)
    p_val = permutation_test(dm, N=100000, order=1, side='right')
    sign = '*' if p_val < 0.05 else ''
    print(f'{name}: {p_val:.5f} {sign}')

print('\n p-Value for Corrected divergence:')
for name in names:
    if key == 'kmer':
        result = np.load(f'data/processed/review/carryover/{name}_k{k}_distance_matrix_c{c}.npz', allow_pickle=True)
    elif key == 'mfe':
        result = np.load(f'data/processed/review/carryover/{name}_distance_matrix_mfe_c{c}.npz', allow_pickle=True)
    distance_matrix_set = result['distance_matrix_set']
    p_val = permutation_test(distance_matrix_set.mean(axis=0), N=100000, order=1, side='right')
    sign = '*' if p_val < 0.05 else ''
    print(f'{name}: {p_val:.5f} {sign}')

p-Value for Original divergence:
N14: 0.00672 *
N14p: 0.01224 *
N20: 0.96040 
N20p: 0.06896 
N14_new: 0.86544 
N14p_new: 0.13049 

 p-Value for Corrected divergence:
N14: 0.01916 *
N14p: 0.02247 *
N20: 0.94754 
N20p: 0.10584 
N14_new: 0.96974 
N14p_new: 0.26761 


#### Response 1-3 & Fig S8: Length distribution

In [ ]:
# Single round simulation parameters

"""
N: Number of initial sequences to generate.
num_reaction: Number of reactions to simulate (N14, N20: 20000; N14p, N20p: 80000).
num_experiments: Number of independent experiments to run.
l: Length of initial sequences.
temp: Temperature in Celsius for RNA folding.
k: Length of minimal complementary regions.
p_cleave: Probability of cleavage at each cleavage site.
gamma: Parameter for caluculating the probability of ligation.
p_cycphos: Probability of cyclic phosphate formation in the cleavage reaction.
p_cycphos_pre: Probability of cyclic phosphate formation in initial sequences (0.0 for N14 and N20, 1.0 for N14p and N20p).
"""

N = 20000
num_reaction = 20000 # or 80000
num_experiments = 10
l = 14
temp = 22.0
k = 6
p_cleave = 0.02
gamma = 2.0
p_cycphos = 1.0
p_cycphos_pre = 0.0 # or 1.0

exp_name = "N14-like2"

In [ ]:
# Perform the single round simulation and save the results

dir = 'data/processed/review/simulation_result'
if not os.path.exists(dir+f'/{exp_name}'):
    os.makedirs(dir+f'/{exp_name}')

for exp in range(num_experiments):
    start = time.time()

    # First population
    seqs = generate_random_sequences(N=N, length=l, base=['A', 'T', 'G', 'C'])
    lens = list(map(len, seqs))
    num_cycphos = int(len(seqs) * p_cycphos_pre)
    labels = ['cyclic-phosphate'] * num_cycphos + ['phosphate'] * (len(seqs) - num_cycphos)
    random.shuffle(labels)
    seqs, lens, labels = pre_cleaving(seqs, lens, labels, p_cleave, p_cycphos)
    counts = np.full(len(seqs), 1, dtype=int)
    df = DataFrame({'seqs': seqs, 'lens': lens, 'counts': counts, 'labels': labels})
    df = df.sample(n=len(df)).reset_index(drop=True)
    df = df[df['lens'] >= k].reset_index(drop=True)
    df = rna_folding(df, temp=temp)
    labeled_sts = [structure_labeling_cofold(st) for st in df['sts']]
    df['labeled_sts'] = labeled_sts


    # Compute possible reaction pairs
    seqs = df['seqs'].values
    seqs_comp = [seq.translate(str.maketrans('ATGC', 'TACG'))[::-1] for seq in seqs]
    S = count_kmer_overlaps(seqs_comp, seqs, k, return_sparse=True, binary=True)
    overlap_regions = extract_overlap_regions(seqs_comp, seqs, S, k=k)
    S_thr, filtered_overlap_regions = overlap_unpairing_filtering(S, seqs, overlap_regions, sts=df['labeled_sts'].to_list(), k=k, temp=temp, unpair_threshold=k, target='both')
    if S_thr.nnz == 0:
        print(f"Experiment {exp+1}: No valid reaction pairs after filtering. Skipping this experiment.")
        continue
    if S_thr.nnz > 5*num_reaction:
        S_thr, filtered_overlap_regions = random_filtering(S_thr, filtered_overlap_regions, N=5*num_reaction, random_state=exp)
    # print(f"After overlap unpairing filtering: number of non-zero entries: {S_thr.nnz}, density: {S_thr.nnz / (S_thr.shape[0] * S_thr.shape[1]):.6f}")


    # Perform the reaction simulation
    df, df_prod = reaction(df, S_thr, filtered_overlap_regions, num_reaction=num_reaction, p_cleave=p_cleave, p_cycphos=p_cycphos, gamma=gamma)


    # Save the results of the experiment
    save_file = {'df': df, 'df_prod': df_prod, 'S': S_thr, 'overlap_regions': filtered_overlap_regions}
    pickle.dump(save_file, open(f"data/processed/review/simulation/{exp_name}/exp_{exp_name}_{exp+1}.pkl", "wb"))
    print(f"Experiment {exp+1} completed in {time.time() - start:.2f} seconds.")

    del df, df_prod, S, S_thr, overlap_regions, filtered_overlap_regions

params = {'N': N, 'num_reaction': num_reaction,
          'num_experiments': num_experiments, 'l': l, 'temp': temp, 'k': k,
          'p_cleave': p_cleave, 'gamma': gamma, 'p_cycphos': p_cycphos,
          'p_cycphos_pre': p_cycphos_pre}

json.dump(params, open(f"data/processed/review/simulation_result/{exp_name}/params_{exp_name}.json", "w"), indent=4)

In [ ]:
# Transfer simulation parameters

"""
Rounds: Number of rounds of transfer simulation.
dil: Dilution rate for each round of transfer simulation.
"""

N = 20000
num_reaction = 80000 # or 100000
num_experiments = 5
l = 20
temp = 22.0
k = 6
p_cleave = 0.02
gamma = 2.0
p_cycphos = 1.0
p_cycphos_pre = 0.0 # or 1.0
Rounds = 30
dil = 0.2

exp_name = "N20p-like-transfer2"

In [ ]:
# Perform the transfer simulation and save the results

dir = 'data/processed/review/simulation_result'
if not os.path.exists(dir+f'/{exp_name}'):
    os.makedirs(dir+f'/{exp_name}')

for R in range(Rounds):
    if not os.path.exists(dir+f'/{exp_name}/R{R+1}'):
        os.makedirs(dir+f'/{exp_name}/R{R+1}')
    for exp in range(num_experiments):
        start = time.time()

        # First population
        seqs = generate_random_sequences(N=N, length=l, base=['A', 'T', 'G', 'C'])
        lens = list(map(len, seqs))
        num_cycphos = int(len(seqs) * p_cycphos_pre)
        labels = ['cyclic-phosphate'] * num_cycphos + ['phosphate'] * (len(seqs) - num_cycphos)
        random.shuffle(labels)
        seqs, lens, labels = pre_cleaving(seqs, lens, labels, p_cleave, p_cycphos)
        counts = np.full(len(seqs), 1, dtype=int)
        df = DataFrame({'seqs': seqs, 'lens': lens, 'counts': counts, 'labels': labels})
        df = df.sample(n=len(df)).reset_index(drop=True)
        df = df[df['lens'] >= k].reset_index(drop=True)

        if R > 0:
            with open(f"data/processed/review/simulation/{exp_name}/R{R}/exp_{exp_name}_{exp+1}.pkl", "rb") as f:
                data = pickle.load(f)
            df_prev = data['df']
            df_prod_prev = data['df_prod']
            # df_prod_prev['labels'] = ['phosphate'] * len(df_prod_prev)
            inds1 = np.where(df_prev['counts'].values > 0)[0]
            inds2 = np.where(df_prod_prev['counts'].values > 0)[0]
            num_prev = len(df)

            seqs_prev = df_prev.iloc[inds1]['seqs'].to_list() + df_prod_prev.iloc[inds2]['seqs'].to_list()
            counts_prev = df_prev.iloc[inds1]['counts'].to_list() + df_prod_prev.iloc[inds2]['counts'].to_list()
            labels_prev = df_prev.iloc[inds1]['labels'].to_list() + df_prod_prev.iloc[inds2]['labels'].to_list()
            lens_prev = df_prev.iloc[inds1]['lens'].to_list() + df_prod_prev.iloc[inds2]['lens'].to_list()
            df_prev = DataFrame({'seqs': seqs_prev, 'lens': lens_prev, 'counts': counts_prev, 'labels': labels_prev})
            df_prev = df_prev.sample(n=len(df_prev)).reset_index(drop=True)

            df_prev = df_prev[:int(np.ceil(num_prev*dil))].reset_index(drop=True)
            df = df[:int(np.ceil(num_prev*(1-dil)))].reset_index(drop=True)
            df = pd.concat([df_prev, df], ignore_index=True)
        
        df = rna_folding(df, temp=temp)
        labeled_sts = [structure_labeling_cofold(st) for st in df['sts']]
        df['labeled_sts'] = labeled_sts


        # Compute possible reaction pairs
        seqs = df['seqs'].values
        seqs_comp = [seq.translate(str.maketrans('ATGC', 'TACG'))[::-1] for seq in seqs]
        S = count_kmer_overlaps(seqs_comp, seqs, k, return_sparse=True, binary=True)
        overlap_regions = extract_overlap_regions(seqs_comp, seqs, S, k=k)
        S_thr, filtered_overlap_regions = overlap_unpairing_filtering(S, seqs, overlap_regions, sts=df['labeled_sts'].to_list(), k=k, temp=temp, unpair_threshold=k, target='both')
        if S_thr.nnz == 0:
            print(f"Round {R+1} Experiment {exp+1}: No valid reaction pairs after filtering. Skipping this experiment.")
            continue
        if S_thr.nnz > 5*num_reaction:
            S_thr, filtered_overlap_regions = random_filtering(S_thr, filtered_overlap_regions, N=5*num_reaction, random_state=exp)
        # print(f"After overlap unpairing filtering: number of non-zero entries: {S_thr.nnz}, density: {S_thr.nnz / (S_thr.shape[0] * S_thr.shape[1]):.6f}")


        # Perform the reaction simulation
        df, df_prod = reaction(df, S_thr, filtered_overlap_regions, num_reaction=num_reaction, p_cleave=p_cleave, p_cycphos=p_cycphos, gamma=gamma)


        # Save the results of the experiment
        save_file = {'df': df, 'df_prod': df_prod, 'S': S_thr, 'overlap_regions': filtered_overlap_regions}
        pickle.dump(save_file, open(f"data/processed/review/simulation/{exp_name}/R{R+1}/exp_{exp_name}_{exp+1}.pkl", "wb"))
        print(f"Round {R+1} Experiment {exp+1} completed in {time.time() - start:.2f} seconds.")

        del df, df_prod, S, S_thr, overlap_regions, filtered_overlap_regions

params = {'N': N, 'num_reaction': num_reaction,
          'num_experiments': num_experiments, 'l': l, 'temp': temp, 'k': k,
          'p_cleave': p_cleave, 'gamma': gamma, 'p_cycphos': p_cycphos,
          'p_cycphos_pre': p_cycphos_pre, 'dil': dil, 'Rounds': Rounds}

json.dump(params, open(f"data/processed/review/simulation/{exp_name}/params_{exp_name}.json", "w"), indent=4)

In [ ]:
# Length distribution calculation (Single round simulation)

exp_name = "N14p-like2"
num_experiments = 10
name = exp_name.split('-')[0]

if name in ['N14', 'N14p']:
    bins = np.arange(16, 37, 1)
elif name in ['N20', 'N20p']:
    bins = np.arange(22, 47, 1)

freqs = []
rs = []
for exp in range(num_experiments):
    with open(f"data/processed/review/simulation/{exp_name}/exp_{exp_name}_{exp+1}.pkl", "rb") as f:
        data = pickle.load(f)
    df_prod = data['df_prod']
    df = data['df']
    r = df_prod['counts'].sum() / df['counts'].sum()
    freq, _ = np.histogram(df_prod['lens'], bins=bins, weights=df_prod['counts'])
    freq = freq.astype(float)
    ###
    freq = 0.75*freq + 0.25*(np.concatenate([freq[1:], [0]]))
    ###
    freq /= freq.sum()
    freqs.append(freq)
    rs.append(r)
freqs = np.array(freqs)
print(f"Mean reaction rate: {np.mean(rs):.4%}")

if name in ['N14', 'N14p']:
    freqs_exp = np.load(f'data/processed/length_distribution/{name}_length_distribution_m15_M45.npy.npy')
    minl_, maxl_ = 15, 45
    freqs_exp = freqs_exp[bins[0]-minl_:bins[-2]-maxl_]
elif name in ['N20', 'N20p']:
    freqs_exp = np.load(f'data/processed/length_distribution/{name}_length_distribution_m21_M45.npy')
    minl_, maxl_ = 21, 45
    freqs_exp = freqs_exp[bins[0]-minl_:]
freqs_exp /= freqs_exp.sum(axis=0)
freqs_exp = freqs_exp.T

result = {'bins': bins[:-1], 'freqs_sim': freqs, 'freqs_exp': freqs_exp, 'reaction_rates': rs}
np.savez_compressed(f'data/processed/review/simulation_length_dist/{name}_results.npz', **result)

In [ ]:
# Length distribution calculation (gamma variation)

exp_names = ['N14-like4', 'N14-like3', 'N14-like2']
gammas = [1.2, 1.6, 2.0]
name = exp_names[0].split('-')[0]
num_experiments = 10

if name in ['N14', 'N14p']:
    bins = np.arange(16, 37, 1)
elif name in ['N20', 'N20p']:
    bins = np.arange(22, 47, 1)

freqs_set = []
for exp_name in exp_names:
    freqs = []
    for exp in range(num_experiments):
        with open(f"data/processed/review/simulation/{exp_name}/exp_{exp_name}_{exp+1}.pkl", "rb") as f:
            data = pickle.load(f)
        df_prod = data['df_prod']
        df = data['df']
        # print(f"Experiment {exp+1}: number of products: {df_prod['counts'].sum()}, Reaction rate: {df_prod['counts'].sum() / df['counts'].sum():.4%}")
        freq, _ = np.histogram(df_prod['lens'], bins=bins, weights=df_prod['counts'])
        freq = freq.astype(float)
        ###
        freq = 0.75*freq + 0.25*(np.concatenate([freq[1:], [0]]))
        ###
        freq /= freq.sum()
        freqs.append(freq)
    freqs_set.append(freqs)

freqs_set = np.array(freqs_set)

result = {'bins': bins[:-1], 'freqs_set': freqs_set, 'gammas': np.array(gammas)}
np.savez_compressed(f'data/processed/review/simulation_length_dist/{name}_gammas_results.npz', **result)

In [ ]:
# Length distribution calculation (Transfer simulation)

exp_name = "N14p-like-transfer2"

params = json.load(open(f"data/processed/review/simulation/{exp_name}/params_{exp_name}.json", "r"))
num_experiments = params['num_experiments']
Rounds = 30
name = exp_name.split('-')[0]

if name in ['N14', 'N14p']:
    bins = np.arange(16, 37, 1)
elif name in ['N20', 'N20p']:
    bins = np.arange(22, 47, 1)

freqs_set = []
for R in np.arange(1, Rounds+1):
    freqs = []
    for exp in range(int(num_experiments)):
        with open(f"data/processed/review/simulation/{exp_name}/R{R}/exp_{exp_name}_{exp+1}.pkl", "rb") as f:
            data = pickle.load(f)
        df_prod = data['df_prod']
        df = data['df']
        # print(f"Experiment {exp+1}: number of products: {df_prod['counts'].sum()}, Reaction rate: {df_prod['counts'].sum() / df['counts'].sum():.4%}")
        freq1, _ = np.histogram(df_prod['lens'], bins=bins, weights=df_prod['counts'])
        freq2, _ = np.histogram(df['lens'], bins=bins, weights=df['counts'])
        freq = freq1 + freq2
        freq = freq.astype(float)
        ###
        freq = 0.75*freq + 0.25*(np.concatenate([freq[1:], [0]]))
        ###
        freq /= freq.sum()
        freqs.append(freq)
    freqs_set.append(freqs)

freqs_set = np.array(freqs_set)

result = {'bins': bins[:-1], 'freqs_set': freqs_set, 'Rounds': np.arange(1, Rounds+1)}
np.savez_compressed(f'data/processed/review/simulation_length_dist/{name}_transfer_results.npz', **result)

#### Response 1-5, 2-3 & Fig S16: Additional complementarity analysis
- k: length of complementarity
- N: number of sampling
- sampli_size: number of sequences sampled from each population
- length_range: length range for filtering sequences
- mfe_range: mfe range for filtering sequences

In [ ]:
length_range = (18, 22)
mfe_range = (-5.0, None)

name = 'N14p'
k = 6
N = 10
sample_size = 100000
terminal = True
unpaired = True
if not terminal:
    unpaired = False

if name == 'N14':
    sub_len_5 = k
    sub_len_3 = k
elif name == 'N14p':
    sub_len_5 = k
    sub_len_3 = k
elif name == 'N20p':
    sub_len_5 = k
    sub_len_3 = k

mat_nnz_set = []
for _ in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=sample_size, length_range=length_range)

    seqs_list = []
    seqs_comp_list = []
    for d in ds:
        d = deploy_df_by_counts(d)
        d = rna_folding(d)
        ###
        if mfe_range[0] is not None:
            d = d[d['mfes'] > mfe_range[0]].reset_index(drop=True)
        if mfe_range[1] is not None:
            d = d[d['mfes'] < mfe_range[1]].reset_index(drop=True)
        ###
        seqs = d['seqs'].values
        seqs_list.append(seqs)

        if terminal:
            if unpaired:

                seqs = d['seqs'].values
                unpair_5 = [st[:sub_len_5].count('.') for st in d['sts'].values]
                unpair_3 = [st[-sub_len_3:].count('.') for st in d['sts'].values]
                valid_inds = np.where((np.array(unpair_5) == sub_len_5) | (np.array(unpair_3) == sub_len_3))[0]
                seqs_comp = []
                for idx in valid_inds:
                    seq = seqs[idx]
                    if (unpair_5[idx] == sub_len_5) & (unpair_3[idx] == sub_len_3):
                        seq = seq[:sub_len_5] + seq[-sub_len_3:]
                    elif unpair_5[idx] == sub_len_5:
                        seq = seq[:sub_len_5]
                    elif unpair_3[idx] == sub_len_3:
                        seq = seq[-sub_len_3:]
                    seq = seq.translate(str.maketrans('ATGC', 'TACG'))[::-1]
                    seqs_comp.append(seq)
                seqs_comp_list.append(seqs_comp)
            else:
                seqs_comp = []
                for seq in seqs:
                    seq = seq[:sub_len_5] + seq[-sub_len_3:]
                    seq = seq.translate(str.maketrans('ATGC', 'TACG'))[::-1]
                    seqs_comp.append(seq)
                seqs_comp_list.append(seqs_comp)
        else:
            seqs_comp = [seq.translate(str.maketrans('ATGC', 'TACG'))[::-1] for seq in seqs]
            seqs_comp_list.append(seqs_comp)

    mat_nnz = np.zeros((len(Rs), len(Rs)))

    for i in range(len(Rs)):
        for j in range(i+1):
        # for j in range(len(Rs)):
            S = count_kmer_overlaps(seqs_list[i], seqs_comp_list[j], k, return_sparse=True, binary=True)
            mat_nnz[i, j] = S.nnz
            if i != j:
                mat_nnz[j, i] = S.nnz
    mat_nnz_set.append(mat_nnz)
    p_val = permutation_test(mat_nnz[1:,1:], N=100000, order=1, side='left', pos='lower', allow_diag=True)
    print(f'Elapsed time for {name} iteration {len(mat_nnz_set)}: {time.time() - start:.4f} seconds, p-value: {p_val:.4e}')

mat_nnz_set = np.array(mat_nnz_set)
result = {
    'mat_nnz': mat_nnz_set,
    'sub_len_5': sub_len_5,
    'sub_len_3': sub_len_3,
    'k': k,
    'N': N,
    'Rs': Rs,
    'unpaired': unpaired,
    'terminal': terminal,
    'sample_size': sample_size
}

length_key = f'O{int(length_range[0])}U{int(length_range[1])}'
if mfe_range[0] is not None and mfe_range[1] is not None:
    mfe_key = f'O{int(mfe_range[0])}U{int(mfe_range[1])}mfe'
elif mfe_range[0] is not None:
    mfe_key = f'O{int(mfe_range[0])}mfe'
elif mfe_range[1] is not None:
    mfe_key = f'U{int(mfe_range[1])}mfe'

np.savez_compressed(f'data/processed/review/ovlp/{name}_overlap_summary_sub{sub_len_5}_{sub_len_3}_k{k}_n{N}_unpaired{unpaired}_terminal{terminal}_{length_key}_{mfe_key}.npz', **result)

#### Response 1-5, 2-3 & Fig S18: Additional analysis for enrichment of sequence motifs
- key: whether to analyze the 3' and 5'end or the entire sequences
- kmer_ori: complementary k-mer of applied sem-random RNAs
- sampling_num: number of sequences to sample from each population
- r: minimal repeat number of complementary motifs
- k: length of terminal regions
- mfe_thr: threshold for MFE
- N: number of sampling iterations

In [ ]:
key = 'termini' # or all
kmer_ori = 'CT' # or 'AG'
sampling_num = 500000
r = 2
k = 8
mfe_thr = -10.0
N = 10

kmer_freqs_set = []

for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name='2mer-no-filter', N=sampling_num, remove_rounds='default', length_range=(23, 30))
    if kmer_ori == 'CT':
        ds = [ds[0], ds[4]]
    elif kmer_ori == 'AG':
        ds = [ds[2], ds[4]]
    else:
        raise ValueError(f"Invalid kmer_ori: {kmer_ori}. Must be 'CT' or 'AG'.")
    
    df = concatenate_dataframes(ds)
    seqs = df['seqs'].values
    pattern = kmer_ori+(r'.{2}'+kmer_ori)*r
    inds = np.array([j for j, seq in enumerate(seqs) if re.search(pattern, seq)])
    df_sub = df.iloc[inds].reset_index(drop=True)
    df_no = df.drop(inds).reset_index(drop=True)

    kmers = make_kmers(bases='ATGC', k=2)
    ds = [df_sub[df_sub['label']==0], df[df['label']==1]]
    ds = [rna_folding(d, temp=22.0) for d in ds]

    kmer_freqs = []
    for kmer in kmers:
        pattern = kmer + r'.{2}' + kmer
        freqs = []
        for i, d in enumerate(ds):
            # if i == 0:
            d = d[d['mfes'] < mfe_thr].reset_index(drop=True)
            counts = d['counts'].values
            seqs = d['seqs'].values

            if key == 'termini':
                pattern_counts = np.array([len(re.findall(pattern, seq[:k])) + len(re.findall(pattern, seq[-k:])) for seq in seqs])
            elif key == 'all':
                pattern_counts = np.array([len(re.findall(pattern, seq)) for seq in seqs])

            inds = np.where(pattern_counts > 0)[0]
            freqs.append((d.iloc[inds]['counts'].sum() + 1)/(d['counts'].sum() + 1)) # Pseudo-counts

        kmer_freqs.append(freqs[0]/freqs[1])
    kmer_freqs_set.append(kmer_freqs)
    print(f'Elapsed time for {kmer_ori} iteration {n+1}/{N}: {time.time() - start:.2f} seconds')
kmer_freqs_set = np.array(kmer_freqs_set)

if kmer_ori == 'CT':
    kmer_ori = 'AGNN'
elif kmer_ori == 'AG':
    kmer_ori = 'CUNN'

result = {
    'kmer_freqs_set': kmer_freqs_set,
    'kmer_ori': kmer_ori,
    'sampling_num': sampling_num,
    'r': r,
    'k': k,
    'mfe_thr': mfe_thr,
    'N': N
}

np.savez_compressed(f'data/processed/review/kmer_freqs/kmer_freqs_{kmer_ori}_{key}.npz', **result)

#### Response 2-4 & Fig S13: Variation in terminal 3-mer frequencies
- k: length of terminal regions
- N: number of sampling iterations

In [ ]:
name = 'N14'# or N14p
N = 10
k = 3

kmers_5 = make_kmers(bases=['A', 'T', 'G', 'C'], k=k)
kmers_3 = make_kmers(bases=['A', 'T', 'G', 'C'], k=k)
kmers_3 = np.array([kmer for kmer in kmers_3 if kmer[-1] != 'A'])

data_5_set = []
data_3_set = []
data_single_5_set = []
data_single_3_set = []

for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name=name, length_range=(18, 30), N=100000)
    ds_single, Rs_single = dataloader(name=f'single_{name}', length_range=(18, 30), N=100000)

    data_5 = np.zeros((len(ds), len(kmers_5)))
    data_3 = np.zeros((len(ds), len(kmers_3)))
    for i in range(len(ds)):
        d = ds[i]
        counts = d['counts']
        sub_seqs_5 = [seq[:k] for seq in d['seqs'].values]
        sub_seqs_3 = [seq[-k:] for seq in d['seqs'].values]

        data_5[i] = [counts[np.array(sub_seqs_5) == kmer].sum()/counts.sum() for kmer in kmers_5]
        data_3[i] = [counts[np.array(sub_seqs_3) == kmer].sum()/counts.sum() for kmer in kmers_3]
    data_5_set.append(data_5)
    data_3_set.append(data_3)

    data_single_5 = np.zeros((len(ds_single), len(kmers_5)))
    data_single_3 = np.zeros((len(ds_single), len(kmers_3)))
    for i in range(len(ds_single)):
        d = ds_single[i]
        counts = d['counts']
        sub_seqs_5 = [seq[:k] for seq in d['seqs'].values]
        sub_seqs_3 = [seq[-k:] for seq in d['seqs'].values]

        data_single_5[i] = [counts[np.array(sub_seqs_5) == kmer].sum()/counts.sum() for kmer in kmers_5]
        data_single_3[i] = [counts[np.array(sub_seqs_3) == kmer].sum()/counts.sum() for kmer in kmers_3]
    data_single_5_set.append(data_single_5)
    data_single_3_set.append(data_single_3)

    print(f'Elapsed time for {name} iteration {n+1}/{N}: {time.time() - start:.4f} seconds')

data_5_set = np.array(data_5_set)
data_3_set = np.array(data_3_set)
data_single_5_set = np.array(data_single_5_set)
data_single_3_set = np.array(data_single_3_set)

result = {
    'data_5_set': data_5_set,
    'data_3_set': data_3_set,
    'data_single_5_set': data_single_5_set,
    'data_single_3_set': data_single_3_set,
    'kmers_5': kmers_5,
    'kmers_3': kmers_3,
    'k': k,
    'N': N,
    'Rs': Rs,
    'Rs_single': Rs_single
}

np.savez_compressed(f'data/processed/review/kmer_freqs_std/{name}_k{k}_kmer_freqs.npz', **result)

#### Response 2-7i & Fig S17C: Length distribution of semi-random + N14p products

In [ ]:
length_range=(15, 45)
ds, Rs = dataloader(name='2mer-no-filter', length_range=length_range, method='none')

freqs_set = np.zeros((len(ds), length_range[1] - length_range[0] + 1))
for i in range(len(ds)):
    d = ds[i]
    counts, lens = d['counts'], d['lens']
    freqs = [counts[lens == l].sum()/counts.sum() for l in range(length_range[0], length_range[1] + 1)]
    freqs_set[i] = freqs

result = {
    'freqs_set': freqs_set,
    'length_range': length_range
}
np.savez_compressed(f'data/processed/review/semi-random_length_dist/{name}_length_distribution.npz', **result)

#### Response 2-7f: Pearson correlation coefficients
name = {N14, N14p, N20, N20p}

In [12]:
name = 'N20p'

distance_matrix_set = np.load(f'data/processed/distance_matrix/{name}_k3_distance_matrix.npy')
distance_matrix = np.mean(distance_matrix_set, axis=0)
distance_matrix = mask_triangle(distance_matrix, upper=False)
for i in range(distance_matrix.shape[0]):
    distance_matrix[i, i] = np.nan
vec_kmer = distance_matrix.flatten()
vec_kmer = vec_kmer[~np.isnan(vec_kmer)]
distance_matrix_set_mfes = np.load(f'data/processed/structure/{name}_distance_matrix.npy')
distance_matrix_mfes = np.mean(distance_matrix_set_mfes, axis=0)
distance_matrix_mfes = mask_triangle(distance_matrix_mfes, upper=False)
for i in range(distance_matrix_mfes.shape[0]):
    distance_matrix_mfes[i, i] = np.nan
vec_mfe = distance_matrix_mfes.flatten()
vec_mfe = vec_mfe[~np.isnan(vec_mfe)]
corr = np.corrcoef(vec_kmer, vec_mfe)[0, 1]
print(corr)

0.1981918906537759


#### Response 2-5 & Table S1: 95% confidence intervals for e_{obs}

In [13]:
from scipy.stats import t as tt


for name in ['N14', 'N14p', 'N20', 'N20p', 'N14_new', 'N14p_new']:
    order = 1
    distance_matrix_set = np.load(f'data/processed/distance_matrix/{name}_k3_distance_matrix.npy')
    diffs = []
    for matrix in distance_matrix_set:
        R = matrix.shape[0]
        CR_counts = order * (2 * R - order - 1) / 2
        NCR_counts = (R - order) * (R - order - 1) / 2
        k = -1
        NCR_ave = np.tril(matrix, k=-(order+1)).sum() / NCR_counts
        CR_ave = (np.tril(matrix, k=k) - np.tril(matrix, k=-(order+1))).sum() / CR_counts
        obs_diff = NCR_ave - CR_ave
        diffs.append(obs_diff)

    diffs = np.array(diffs)

    mu = diffs.mean()
    se = diffs.std(ddof=1) / np.sqrt(len(diffs))

    t = tt.ppf(0.975, df=len(diffs)-1)
    print(f'{name} 95% confidence interval: {mu - t*se:.6f} to {mu + t*se:.6f}')

N14 95% confidence interval: 0.007156 to 0.007584
N14p 95% confidence interval: 0.015242 to 0.015761
N20 95% confidence interval: -0.007016 to -0.006607
N20p 95% confidence interval: 0.010185 to 0.010724
N14_new 95% confidence interval: 0.000061 to 0.000494
N14p_new 95% confidence interval: 0.001656 to 0.001830
